### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="forensic_glass_identification",
    dataset_year="1987",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5WW2P",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/42/glass+identification.zip && unzip glass+identification.zip && rm glass+identification.zip glass.names glass.tag Index
mkdir -p local-data-warehouse/forensic_glass_identification && mv glass.data local-data-warehouse/forensic_glass_identification/
""",
    # References
    academic_reference_bibtex="""@misc{German1987glass,
  author       = {German, B.},
  title        = {{Glass Identification}},
  year         = {1987},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C5WW2P}
}
""",
    academic_reference_bibtex_key="German1987glass",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We drop the ID column, as it is a just a row identifier and does not contain any useful information for modeling.
- The data contains one naturally occurring duplicate.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Type_of_glass",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Type_of_glass",
)

## Preprocessing

In [2]:
import pandas as pd

columns = [
    "Id",
    "RI",
    "Na",
    "Mg",
    "Al",
    "Si",
    "K",
    "Ca",
    "Ba",
    "Fe",
    "Type_of_glass"
]

df = pd.read_csv(dataset_mold.path / "glass.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.drop(columns=["Id"])

as_cat_type = ["Type_of_glass"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (214, 11)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 214
Columns: 10
Use sampling: False (sample size: 214)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['RI', 'Ca', 'Na', 'Si', 'Al', 'Mg', 'K', 'Ba', 'Fe']
Rows remaining as candidates after top-9 filter: 2 (of 214)

#### Duplicate Report
Total duplicate rows: 1 (0.47% of dataset)
Duplicate rows ignoring target: 1 (0.47% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type_of_glass
0,1.51755,13.00,3.60,1.36,72.99,0.57,8.40,0.00,0.11,1
1,1.51727,14.70,0.00,2.34,73.28,0.00,8.95,0.66,0.00,7
2,1.52152,13.05,3.65,0.87,72.22,0.19,9.85,0.00,0.17,1
3,1.51602,14.85,0.00,2.38,73.28,0.00,8.76,0.64,0.09,7
4,1.51708,13.72,3.68,1.81,72.06,0.64,7.88,0.00,0.00,2


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Type_of_glass,category,0.0,0.0,6.0,"2, 1, 7, 3, 5, 6"
1,RI,float64,0.0,0.0,178.0,"1.5159, 1.5215, 1.5165, 1.5161, 1.5176, 1.5175, 1.5183, 1.5178, 1.5167, 1.5197"
2,Na,float64,0.0,0.0,142.0,"13.0, 13.02, 13.21, 13.24, 13.64, 12.85, 13.33, 12.86, 13.41, 12.93"
3,Mg,float64,0.0,0.0,94.0,"0.0, 3.54, 3.48, 3.58, 3.52, 3.62, 3.57, 3.56, 3.61, 3.5"
4,Al,float64,0.0,0.0,118.0,"1.54, 1.19, 1.29, 1.43, 1.56, 1.23, 1.28, 1.36, 1.35, 1.63"
5,Si,float64,0.0,0.0,133.0,"72.99, 73.28, 72.86, 73.11, 73.1, 72.95, 72.72, 73.08, 72.97, 73.21"
6,K,float64,0.0,0.0,65.0,"0.0, 0.57, 0.56, 0.6, 0.58, 0.61, 0.64, 0.59, 0.62, 0.54"
7,Ca,float64,0.0,0.0,143.0,"8.03, 8.43, 8.44, 9.57, 8.79, 8.39, 9.85, 8.6, 8.53, 8.67"
8,Ba,float64,0.0,0.0,34.0,"0.0, 0.64, 1.57, 0.09, 0.11, 1.59, 0.66, 0.61, 1.64, 0.76"
9,Fe,float64,0.0,0.0,32.0,"0.0, 0.17, 0.24, 0.09, 0.1, 0.11, 0.14, 0.28, 0.16, 0.12"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
RI,214.0,1.518365,0.003037,1.51115,1.53393
Na,214.0,13.407850,0.816604,10.73000,17.38000
Mg,214.0,2.684533,1.442408,0.00000,4.49000
Al,214.0,1.444907,0.499270,0.29000,3.50000
Si,214.0,72.650935,0.774546,69.81000,75.41000
K,214.0,0.497056,0.652192,0.00000,6.21000
Ca,214.0,8.956963,1.423153,5.43000,16.19000
Ba,214.0,0.175047,0.497219,0.00000,3.15000
Fe,214.0,0.057009,0.097439,0.00000,0.51000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                    
Type_of_glass 1        2     76  35.51
              2        1     70  32.71
              3        7     29  13.55
              4        3     17   7.94
              5        5     13   6.07

In [8]:
# Target Distribution
target_df

,count,pct
Type_of_glass,,
2,76,35.51
1,70,32.71
7,29,13.55
3,17,7.94
5,13,6.07
6,9,4.21


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to forensic_glass_identification/019d5dc1-024d-7ef3-b5fc-d963c3d7e7ca


019d5dc1-024d-7ef3-b5fc-d963c3d7e7ca
dd0021200e23b932da2306196da4f9d3d6ea5f556f9d6f58b37cd8fdd0c9a0ea
